## Territory Sales Growth Analysis: Q4-2021 vs Q3-2021

Write a query to return Territory and corresponding Sales Growth. Compare growth between periods Q4-2021 vs Q3-2021. If Territory (say T123) has Sales worth $100 in Q3-2021 and Sales worth $110 in Q4-2021, then the Sales Growth will be 10% [ i.e. = ((110 - 100)/100) * 100 ]

Output the ID of the Territory and the Sales Growth. Only output these territories that had any sales in both quarters.

🌀By solving this, you'll learn how to use Cte, Group by, Join, Agg function. Give it a try and share the output! 👇

In [0]:
%skip
CREATE TABLE ska_catalog2.bronze.fct_customer_sale (cust_id VARCHAR(50), prod_sku_id VARCHAR(50), order_date TIMESTAMP, order_value BIGINT, order_id VARCHAR(50));

CREATE TABLE ska_catalog2.bronze.map_customer_territories (cust_id VARCHAR(50), territory_id VARCHAR(50));

INSERT INTO ska_catalog2.bronze.fct_customer_sale (cust_id, prod_sku_id, order_date, order_value, order_id) VALUES ('C001', 'P100', '2021-07-15', 100, 'O1001'), ('C002', 'P101', '2021-07-20', 200, 'O1002'), ('C001', 'P100', '2021-10-05', 150, 'O1003'), ('C002', 'P101', '2021-10-10', 250, 'O1004'), ('C003', 'P102', '2021-08-22', 180, 'O1005'), ('C003', 'P102', '2021-11-30', 210, 'O1006');

INSERT INTO ska_catalog2.bronze.map_customer_territories (cust_id, territory_id) VALUES  ('C001', 'T001'), ('C002', 'T002'), ('C003', 'T003');

In [0]:
SELECT * FROM ska_catalog2.bronze.fct_customer_sale;

In [0]:
SELECT * FROM ska_catalog2.bronze.map_customer_territories;

In [0]:
SELECT 
    mt.territory_id,
    QUARTER(fcs.order_date) AS `quarter`,
    YEAR (fcs.order_date) AS `year`,
    SUM(fcs.order_value) AS total_sales
  FROM ska_catalog2.bronze.fct_customer_sale fcs
  JOIN ska_catalog2.bronze.map_customer_territories mt ON fcs.cust_id = mt.cust_id
  WHERE fcs.order_date BETWEEN '2021-07-01' AND '2021-12-31'
  GROUP BY mt.territory_id,     QUARTER(fcs.order_date),
    YEAR(fcs.order_date)

In [0]:
WITH QuarterlySales AS (
  SELECT
    mt.territory_id,
    QUARTER(fcs.order_date) AS `quarter`,
    YEAR (fcs.order_date) AS `year`,
    SUM(fcs.order_value) AS total_sales
  FROM ska_catalog2.bronze.fct_customer_sale fcs
  JOIN ska_catalog2.bronze.map_customer_territories mt ON fcs.cust_id = mt.cust_id
  WHERE fcs.order_date BETWEEN '2021-07-01' AND '2021-12-31'
  GROUP BY mt.territory_id,     QUARTER(fcs.order_date),
    YEAR(fcs.order_date)
)
SELECT q4.territory_id,
  (q4.total_sales - q3.total_sales) * 100.0 / q3.total_sales AS `sales_growth_percentage`
FROM QuarterlySales q4
JOIN QuarterlySales q3
  ON q4.territory_id = q3.territory_id
  AND q4.year = 2021 AND q4.quarter = 4
  AND q3.year = 2021 AND q3.quarter = 3
WHERE q3.total_sales > 0

In [0]:
%python
import pandas as pd

# ska_catalog2.bronze.fct_customer_sale fcs ska_catalog2.bronze.map_customer_territories
df_fct_customer_sale = spark.table('ska_catalog2.bronze.fct_customer_sale').toPandas()
df_map_customer_territories = spark.table('ska_catalog2.bronze.map_customer_territories').toPandas()

if not pd.api.types.is_datetime64_any_dtype(df_fct_customer_sale['order_date']):
    df_fct_customer_sale['order_date'] = pd.to_datetime(df_fct_customer_sale['order_date'])

df_fct_customer_sale['order_date'] = pd.to_datetime(df_fct_customer_sale['order_date'])
df_filtered = df_fct_customer_sale[
    (df_fct_customer_sale['order_date'] >= pd.to_datetime('2021-07-01')) &
    (df_fct_customer_sale['order_date'] <= pd.to_datetime('2021-12-31'))
]
df_merged = df_filtered.merge(df_map_customer_territories, on='cust_id', how='inner')
df_merged['quarter'] = df_merged['order_date'].dt.quarter
df_merged['year'] = df_merged['order_date'].dt.year
result = df_merged.groupby(['territory_id', 'quarter', 'year'], as_index=False)['order_value'].sum()
result.rename(columns={'order_value': 'total_sales'}, inplace=True)


q4 = result[(result['year'] == 2021) & (result['quarter'] == 4)]
q3 = result[(result['year'] == 2021) & (result['quarter'] == 3)]

merged = q4.merge(q3, on='territory_id', suffixes=('_q4', '_q3'))
merged = merged[merged['total_sales_q3'] > 0]
merged['sales_growth_percentage'] = (merged['total_sales_q4'] - merged['total_sales_q3']) * 100.0 / merged['total_sales_q3']

display(merged[['territory_id', 'sales_growth_percentage']])
